# Formative 2: Tabular Data Preprocessing

This notebook explores and combines the supplied customer transaction and social-profile datasets to prepare a transaction-level dataset for predicting `product_category`.

**Important:** social IDs appear to use an `A` prefix (such as `A151`) while transaction IDs are numeric (such as `151`). Removing the prefix is treated as a *candidate mapping*, not a confirmed fact. Verify it against the dataset documentation or facilitator before relying on it.

## 1. Imports and input files

Put the two CSV files in `data/raw/` or in the repository root. The loader checks both locations.

In [4]:
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
ROOT = Path.cwd()

def find_file(names):
    candidates = [ROOT / "data" / "raw" / n for n in names] + [ROOT / n for n in names]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find: " + ", ".join(names))

tx_path = find_file(["customer_transactions.csv",
                     "customer_transactions - customer_transactions.csv"])
social_path = find_file(["customer_social_profiles.csv",
                         "customer_social_profiles - customer_social_profiles.csv"])

transactions = pd.read_csv(tx_path)
social = pd.read_csv(social_path)
print("Transactions:", transactions.shape, tx_path)
print("Social profiles:", social.shape, social_path)
display(transactions.head())
display(social.head())

ModuleNotFoundError: No module named 'matplotlib'

## 2. Initial data audit

Before merging, inspect each table's size, column types, missing values, duplicate rows, and the target distribution.

In [ ]:
def audit(df, name):
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    display(df.dtypes.rename("dtype").to_frame())
    display(df.isna().sum().rename("missing values").to_frame())
    print("Exact duplicate rows:", df.duplicated().sum())
    display(df.head())

audit(transactions, "Transactions")
audit(social, "Social profiles")

print("Product category distribution:")
display(transactions["product_category"].value_counts(dropna=False).to_frame("count"))
transactions["product_category"].value_counts().plot(kind="bar", title="Product category counts")
plt.xlabel("Product category")
plt.ylabel("Transactions")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 3. Validate columns and clean data types

The transaction table is the primary table because each row represents a transaction. Rows without a customer ID or target label cannot be used in the supervised modelling dataset.

In [ ]:
required_tx = {"customer_id_legacy", "transaction_id", "purchase_amount",
               "purchase_date", "product_category", "customer_rating"}
required_social = {"customer_id_new", "social_media_platform", "engagement_score",
                   "purchase_interest_score", "review_sentiment"}

assert required_tx.issubset(transactions.columns), f"Missing transaction columns: {required_tx - set(transactions.columns)}"
assert required_social.issubset(social.columns), f"Missing social columns: {required_social - set(social.columns)}"

tx = transactions.copy()
sp = social.copy()

tx["customer_id_legacy"] = tx["customer_id_legacy"].astype("string").str.strip()
sp["customer_id_new"] = sp["customer_id_new"].astype("string").str.strip()
tx["purchase_date"] = pd.to_datetime(tx["purchase_date"], errors="coerce")
tx["product_category"] = tx["product_category"].astype("string").str.strip()

for col in ["purchase_amount", "customer_rating"]:
    tx[col] = pd.to_numeric(tx[col], errors="coerce")
for col in ["engagement_score", "purchase_interest_score"]:
    sp[col] = pd.to_numeric(sp[col], errors="coerce")

print("Missing target rows:", tx["product_category"].isna().sum())
print("Invalid purchase dates:", tx["purchase_date"].isna().sum())
print("Missing transaction customer IDs:", tx["customer_id_legacy"].isna().sum())
print("Missing social customer IDs:", sp["customer_id_new"].isna().sum())

tx = tx.dropna(subset=["customer_id_legacy", "product_category"]).copy()

## 4. Investigate the customer-ID relationship

We test whether removing one leading `A` from `customer_id_new` creates overlap with `customer_id_legacy`. The counts below show how much of the data would match under this assumption.

In [ ]:
sp["customer_id_candidate"] = sp["customer_id_new"].str.replace(r"^A", "", regex=True)
tx_ids = set(tx["customer_id_legacy"].dropna().astype(str))
social_ids = set(sp["customer_id_candidate"].dropna().astype(str))
overlap = tx_ids & social_ids

print("Unique transaction customers:", len(tx_ids))
print("Unique social customers after candidate normalization:", len(social_ids))
print("Overlapping IDs:", len(overlap))
print("Transaction IDs without social match:", len(tx_ids - social_ids))
print("Social IDs without transaction match:", len(social_ids - tx_ids))
print("Example matches:", sorted(overlap)[:10])

## 5. Aggregate repeated social records

Both tables contain repeated customer IDs. Transactions are kept at transaction level. Social records are summarized to one row per customer before joining. This prevents a many-to-many join from multiplying transaction rows.

In [ ]:
def stable_mode(values):
    values = values.dropna().astype(str)
    if values.empty:
        return pd.NA
    counts = values.value_counts()
    return sorted(counts[counts == counts.max()].index)[0]

social_customer = sp.groupby("customer_id_candidate").agg(
    social_record_count=("customer_id_new", "size"),
    distinct_platform_count=("social_media_platform", "nunique"),
    engagement_mean=("engagement_score", "mean"),
    engagement_max=("engagement_score", "max"),
    engagement_std=("engagement_score", "std"),
    purchase_interest_mean=("purchase_interest_score", "mean"),
    purchase_interest_max=("purchase_interest_score", "max"),
    purchase_interest_std=("purchase_interest_score", "std"),
    primary_platform=("social_media_platform", stable_mode),
    dominant_sentiment=("review_sentiment", stable_mode),
).reset_index()

platform_flags = pd.crosstab(
    sp["customer_id_candidate"], sp["social_media_platform"]
).gt(0).astype("int8")
platform_flags.columns = [
    "platform_" + str(c).strip().lower().replace(" ", "_")
    for c in platform_flags.columns
]
platform_flags = platform_flags.reset_index()
social_customer = social_customer.merge(
    platform_flags, on="customer_id_candidate", validate="one_to_one"
)

print("Social records:", len(sp))
print("Customer-level social rows:", len(social_customer))
print("Unique customer keys:", social_customer["customer_id_candidate"].is_unique)
display(social_customer.head())

## 6. Merge and validate

A left join keeps every valid transaction, including customers without a matching social profile. `validate='many_to_one'` checks that the aggregated social table has no more than one row per join key.

In [ ]:
merged = tx.merge(
    social_customer,
    left_on="customer_id_legacy",
    right_on="customer_id_candidate",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Transaction rows before merge:", len(tx))
print("Rows after merge:", len(merged))
print("Matched transaction rows:", (merged["_merge"] == "both").sum())
print("Unmatched transaction rows:", (merged["_merge"] == "left_only").sum())
print("Unique transaction IDs after merge:", merged["transaction_id"].nunique())

assert len(merged) == len(tx), "The merge changed the transaction row count."
assert merged["transaction_id"].is_unique, "Transaction IDs are duplicated after merge."

merged = merged.drop(columns=["customer_id_candidate", "_merge"])

## 7. Feature preparation and target checks

The target is `product_category`. Transaction identifiers and customer identifiers are excluded from baseline predictors. Purchase amount and customer rating are also excluded because they may only be known after a purchase, depending on the intended prediction scenario.

In [ ]:
target = "product_category"
merged["purchase_year"] = merged["purchase_date"].dt.year
merged["purchase_month"] = merged["purchase_date"].dt.month
merged["purchase_dayofweek"] = merged["purchase_date"].dt.dayofweek

excluded = {
    "transaction_id", "customer_id_legacy", "customer_id_candidate",
    "purchase_date", "purchase_amount", "customer_rating", target
}
feature_columns = [c for c in merged.columns if c not in excluded]

model_table = merged[feature_columns + [target]].copy()
print("Feature columns:", feature_columns)
print("Target classes:", sorted(model_table[target].dropna().astype(str).unique()))
display(model_table.head())

class_counts = model_table[target].value_counts()
display(pd.concat([
    class_counts.rename("count"),
    (class_counts / class_counts.sum() * 100).round(1).rename("percent")
], axis=1))

social_cols = [c for c in social_customer.columns if c != "customer_id_candidate"]
display(merged[social_cols].isna().mean().sort_values(ascending=False)
        .rename("missing_fraction").to_frame())

## 8. Save processed datasets

The raw CSV files are never overwritten. The outputs are saved to `data/processed/` for later modelling.

In [ ]:
output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

tx.to_csv(output_dir / "transactions_clean.csv", index=False)
social_customer.to_csv(output_dir / "social_profiles_customer_level.csv", index=False)
merged.to_csv(output_dir / "customer_transactions_social_merged.csv", index=False)
model_table.to_csv(output_dir / "tabular_model_features.csv", index=False)

print("Saved processed files:")
for path in sorted(output_dir.glob("*.csv")):
    print("-", path)

## 9. Final validation summary

These assertions make sure the merge has not multiplied transactions and that the target is available for supervised learning.

In [ ]:
summary = pd.Series({
    "input_transaction_rows": len(transactions),
    "transaction_rows_after_cleaning": len(tx),
    "merged_transaction_rows": len(merged),
    "unique_transaction_ids_after_merge": merged["transaction_id"].nunique(),
    "unique_transaction_customers": merged["customer_id_legacy"].nunique(),
    "social_customer_rows_after_aggregation": len(social_customer),
    "social_customer_keys_unique": social_customer["customer_id_candidate"].is_unique,
    "missing_target_after_cleaning": merged[target].isna().sum(),
    "matched_transaction_rows": int(merged[social_cols].notna().any(axis=1).sum()),
    "unmatched_transaction_rows": int(merged[social_cols].isna().all(axis=1).sum()),
})
display(summary.to_frame("value"))

assert len(merged) == len(tx)
assert merged["transaction_id"].is_unique
assert social_customer["customer_id_candidate"].is_unique
assert merged[target].notna().all()
print("Tabular preprocessing completed.")

## 10. Scope and next steps

This notebook completes the tabular preprocessing stage only. It does not create artificial facial images or voice recordings. The remaining assignment stages require genuine participant data collected with informed consent.

- **Images:** collect neutral, smiling, and surprised samples per participant; augment training images; extract image features; avoid identity leakage by splitting data by person.
- **Audio:** collect at least two phrases per participant; inspect waveforms and spectrograms; augment training audio; extract MFCC, spectral roll-off, and energy; split by speaker.
- **Models:** train and evaluate face recognition, voice verification, and product-category prediction separately before integrating their decisions into the transaction simulation.